In [ ]:
from pathlib import Path
from mhdr.dataloader.io import read_csv, save_csv, load_dimension_sets
import pandas as pd
from mhdr.pipeline.semantic_mapper import SemanticMapper
from mhdr.pipeline.seed_retriever import retrieve_seeds
from itertools import combinations
from pathlib import Path
from datetime import datetime
import gc
import pandas as pd
import torch
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"
DIMENSIONS = [
    "Emotional",
    "Environmental",
    "Financial",
    "Intellectual",
    "Occupational",
    "Physical",
    "Social",
    "Spiritual",
]
questions = read_csv(INPUT_DIR / "questions.csv")
dimension_sets = load_dimension_sets(INPUT_DIR / "dim_definations.csv")
print(questions.head())
print(dimension_sets.keys())


/Users/haikeyu/Desktop/mentalhealth-dimension-reduction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


         qid                                       text  dataset
0  CD_RISC_1     I am able to adapt when changes occur.  CD-RISC
1  CD_RISC_2  I have one close and secure relationship.  CD-RISC
2  CD_RISC_3            Sometimes fate or God helps me.  CD-RISC
3  CD_RISC_4     I can deal with whatever comes my way.  CD-RISC
4  CD_RISC_5         Past successes give me confidence.  CD-RISC
dict_keys(['ChatGPT-5.2', 'DeepSeek-V3.2', 'Llama-4', 'claude-sonnet-4.5', 'gemini-3.0-pro'])


In [2]:
mapper = SemanticMapper()
mapper.set_questions_df(questions, text_col="text", qid_col="qid", dataset_col="dataset")

mean_df = mapper.score_and_average(dimension_sets)
save_csv(mean_df, TEMP_DIR / "semantic_mapping_results.csv")
print(mean_df.head())


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1663.97it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


          qid                                               text  dataset  \
0   CD_RISC_1             I am able to adapt when changes occur.  CD-RISC   
1  CD_RISC_10             I make my best effort, no matter what.  CD-RISC   
2  CD_RISC_11  I believe I can achieve my goals, even if ther...  CD-RISC   
3  CD_RISC_12              Even when hopeless, I do not give up.  CD-RISC   
4  CD_RISC_13     In times of stress, I know where to find help.  CD-RISC   

   Emotional  Environmental  Financial  Intellectual  Occupational  Physical  \
0   0.174857       0.072354   0.073339      0.164429      0.126643  0.089954   
1   0.148006       0.090310   0.150222      0.259957      0.230811  0.172987   
2   0.231878       0.154928   0.256694      0.276000      0.248534  0.190872   
3   0.244273       0.022185   0.181059      0.151116      0.158120  0.031285   
4   0.360147       0.150452   0.169271      0.130460      0.121779  0.159801   

     Social  Spiritual  
0  0.110292   0.148546  
1  0.1

In [3]:
seeds = retrieve_seeds(mean_df, DIMENSIONS, ["Emotional", "Social"], k=5)
seed_texts = seeds["text"].tolist()
print(seed_texts)

['In the past, I have not always had friends with whom I could share my joys and sorrows.', 'I have not experienced many warm and trusting relationships with others.', 'Maintaining close relationships has been difficult and frustrating for me.', 'It is difficult for me to make friends', 'There have been times when I felt inferior to most of the people I knew.']


In [4]:
import gc
import torch

print("=" * 60)
print("PRE-CLEAN MEMORY")
print("=" * 60)

# delete large objects if they exist
for var in [
    "gen",
    "single_df",
    "pair_df",
    "triple_df",
    "ALL_RESULTS",
    "GENERATED_TEXT_HISTORY",
]:
    if var in globals():
        del globals()[var]

# python garbage collection
gc.collect()

# clear GPU memory if using cuda
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Memory cleaned.")
print("=" * 60)

PRE-CLEAN MEMORY
Memory cleaned.


In [5]:
from mhdr.pipeline.generator import SeededQuestionGenerator

gen = SeededQuestionGenerator(model_name="google/flan-t5-large")

df_gen = gen.generate_to_df(
    target_dims=["Emotional", "Social"],
    seed_texts=seed_texts,
    n_questions=30,
)

print(df_gen.head(10))

Loading weights: 100%|██████████| 558/558 [00:02<00:00, 226.42it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


         target_dims                                     generated_text
0  Emotional, Social  I have been able to share my joys and sorrows ...
1  Emotional, Social  I have had a few friends whom I could talk wit...
2  Emotional, Social  I am unable to make friends because I have had...
3  Emotional, Social  I was worried about people, or other people ar...
4  Emotional, Social  I am able to talk with people more easily than...
5  Emotional, Social  I enjoy having friends who could understand me...
6  Emotional, Social  Having an intimate relationship with people ha...
7  Emotional, Social  I would never have friends in my life that I c...
8  Emotional, Social  I often feel lonely with no one to share my jo...
9  Emotional, Social  I have not always been able to make friends wi...


In [6]:
OUTPUT_DIR.mkdir(exist_ok=True)

SINGLE_PATH = OUTPUT_DIR / "generated_questions_single.csv"
PAIR_PATH = OUTPUT_DIR / "generated_questions_pair.csv"
TRIPLE_PATH = OUTPUT_DIR / "generated_questions_triple.csv"

gen = SeededQuestionGenerator(model_name="google/flan-t5-base")

GENERATED_TEXT_HISTORY = []

print("=" * 60)
print("START GENERATION PIPELINE")
print("Time:", datetime.now())
print("=" * 60)

SINGLE_COMBOS = [[dim] for dim in DIMENSIONS]
PAIR_COMBOS = [list(c) for c in combinations(DIMENSIONS, 2)]
TRIPLE_COMBOS = [list(c) for c in combinations(DIMENSIONS, 3)]

print(f"Single combos: {len(SINGLE_COMBOS)}")
print(f"Pair combos:   {len(PAIR_COMBOS)}")
print(f"Triple combos: {len(TRIPLE_COMBOS)}")


def run_generation_for_combo(target_dims, idx, total, n_seeds=6, n_questions=30):
    target_dims = sorted(target_dims)

    print("\n" + "-" * 60)
    print(f"[{idx}/{total}] TARGET: {target_dims}")
    print("-" * 60)

    seeds = retrieve_seeds(
        mean_df,
        DIMENSIONS,
        target_dims,
        k=n_seeds,
    )

    if seeds.empty:
        print("No valid seeds. Skipping.")
        return None

    seed_texts = seeds["text"].tolist()

    print(f"Seeds found: {len(seed_texts)}")
    for j, s in enumerate(seed_texts, 1):
        print(f"[{j}] {s}")

    print("Generating...")
    df_gen = gen.generate_to_df(
        target_dims=target_dims,
        seed_texts=seed_texts,
        n_questions=n_questions,
        existing_texts=GENERATED_TEXT_HISTORY,
        min_words=10,
        ngram_n=2,
        jaccard_threshold=0.5,
        batch_size=3,
        max_new_tokens=32,
        temperature=0.9,
        top_p=0.95,
        max_total_batches=1000,
        require_exact_count=True,
    )

    n_generated = len(df_gen)
    print(f"Generated samples: {n_generated}")

    if n_generated == 0:
        print("Generation failed. Skipping.")
        return None

    df_gen["target_dims"] = ", ".join(target_dims)
    df_gen["combo_size"] = len(target_dims)
    df_gen["target_count"] = n_questions
    df_gen["generated_count"] = n_generated

    for j in range(n_seeds):
        col = f"seed_{j+1}"
        df_gen[col] = seed_texts[j] if j < len(seed_texts) else None

    return df_gen


def run_group(combos, save_path, group_name, n_seeds=6, n_questions=30):
    print("\n" + "=" * 60)
    print(f"RUNNING GROUP: {group_name.upper()}")
    print("=" * 60)

    group_results = []

    for i, combo in enumerate(combos, 1):
        df = run_generation_for_combo(
            target_dims=combo,
            idx=i,
            total=len(combos),
            n_seeds=n_seeds,
            n_questions=n_questions,
        )

        if df is not None:
            group_results.append(df)
            GENERATED_TEXT_HISTORY.extend(df["generated_text"].astype(str).tolist())

            group_progress = pd.concat(group_results, ignore_index=True)
            group_progress.to_csv(save_path, index=False)
            print(f"Saved group progress to: {save_path}")
            print(f"Current rows in {group_name}: {len(group_progress)}")
            del group_progress

        if df is not None:
            del df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if len(group_results) == 0:
        print(f"No results generated for {group_name}.")
        return None

    group_df = pd.concat(group_results, ignore_index=True)
    group_df.to_csv(save_path, index=False)

    print("\n" + "-" * 60)
    print(f"{group_name.upper()} COMPLETE")
    print(f"Saved to: {save_path}")
    print(f"Total rows: {len(group_df)}")
    print("-" * 60)

    return group_df


single_df = run_group(
    combos=SINGLE_COMBOS,
    save_path=SINGLE_PATH,
    group_name="single",
    n_seeds=6,
    n_questions=30,
)



Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1647.94it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


START GENERATION PIPELINE
Time: 2026-03-20 02:39:29.784056
Single combos: 8
Pair combos:   28
Triple combos: 56

RUNNING GROUP: SINGLE

------------------------------------------------------------
[1/8] TARGET: ['Emotional']
------------------------------------------------------------
Seeds found: 6
[1] I am able to handle unpleasant or painful feelings like sadness, fear, and anger.
[2] How often do you feel sad?
[3] How often do you feel angry?
[4] How often do you feel anxious?
[5] I notice when stress begins to affect my body.
[6] In times of stress, I know where to find help.
Generating...
Generated samples: 30
Saved group progress to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_single.csv
Current rows in single: 30

------------------------------------------------------------
[2/8] TARGET: ['Environmental']
------------------------------------------------------------
Seeds found: 6
[1] My environment makes it easier for me

In [7]:
pair_df = run_group(
    combos=PAIR_COMBOS,
    save_path=PAIR_PATH,
    group_name="pair",
    n_seeds=6,
    n_questions=30,
)



RUNNING GROUP: PAIR

------------------------------------------------------------
[1/28] TARGET: ['Emotional', 'Environmental']
------------------------------------------------------------
Seeds found: 6
[1] Noise or clutter in my environment makes it harder for me to relax.
[2] My environment makes it easier for me to take care of myself.
[3] I feel less stressed when my surroundings are quiet and orderly.
[4] I am able to handle unpleasant or painful feelings like sadness, fear, and anger.
[5] The condition of my surroundings affects my mood.
[6] I feel more at peace in spaces that are calm and familiar.
Generating...
Generated samples: 30
Saved group progress to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_pair.csv
Current rows in pair: 30

------------------------------------------------------------
[2/28] TARGET: ['Emotional', 'Financial']
------------------------------------------------------------
Seeds found: 6
[1] I fe

In [8]:

triple_df = run_group(
    combos=TRIPLE_COMBOS,
    save_path=TRIPLE_PATH,
    group_name="triple",
    n_seeds=6,
    n_questions=30,
)



RUNNING GROUP: TRIPLE

------------------------------------------------------------
[1/56] TARGET: ['Emotional', 'Environmental', 'Financial']
------------------------------------------------------------
Seeds found: 6
[1] My financial situation affects how secure I feel.
[2] Unexpected expenses create stress for me.
[3] I feel more secure when I am able to save for the future.
[4] During the past month, how often have you had trouble sleeping because you cannot breathe comfortably?
[5] Money concerns interfere with my peace of mind.
[6] In times of stress, I know where to find help.
Generating...
Generated samples: 30
Saved group progress to: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_triple.csv
Current rows in triple: 30

------------------------------------------------------------
[2/56] TARGET: ['Emotional', 'Environmental', 'Intellectual']
------------------------------------------------------------
Seeds found: 6
[1] I f

In [9]:

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

if single_df is not None:
    print(f"Single file: {SINGLE_PATH}")
    print(f"Single rows: {len(single_df)}")

if pair_df is not None:
    print(f"Pair file:   {PAIR_PATH}")
    print(f"Pair rows:   {len(pair_df)}")

if triple_df is not None:
    print(f"Triple file: {TRIPLE_PATH}")
    print(f"Triple rows: {len(triple_df)}")

print("\nDONE")
print("Time:", datetime.now())


FINAL SUMMARY
Single file: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_single.csv
Single rows: 240
Pair file:   /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_pair.csv
Pair rows:   840
Triple file: /Users/haikeyu/Desktop/mentalhealth-dimension-reduction/laboratory/07_mhdr/output/generated_questions_triple.csv
Triple rows: 1680

DONE
Time: 2026-03-20 09:26:35.199924
